# 실습 7 · 사전학습 모델로 사진 판별하기

> **실습본입니다.** 먼저 메뉴 **파일 → Drive에 사본 저장**을 누르세요. 사본을 저장하지 않으면 입력한 코드가 사라집니다.
> 맨 위 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다. ▶ 셀은 그대로 실행하고, ✍️ 셀은 위에 보여 준 코드를
> 빈 셀에 손으로 타이핑하고, 빈칸(`___`)은 채워서 실행합니다. 📝 과제는 힌트를 보고 스스로 풉니다.

**⏱ 75분** · 7주차 실습 — ImageNet으로 학습된 ResNet-18을 가져와 학습 없이 사진을 판별하고, 마지막 층을 바꿔 개미·벌 사진 244장으로 다시 학습하기

## 🎯 이번 시간의 질문

실습 6에서는 CIFAR-10 사진 4만 장으로 CNN을 처음부터 학습시켰고, 고양이 예시 사진의 판별이 학습할 때마다 흔들렸습니다.

> **이미 학습된 모델을 가져오면, 적은 사진으로도 맞힐까요?**

ImageNet 사진 약 128만 장으로 학습된 ResNet-18을 내려받아 그대로 써 보고, 분류부만 갈아 끼워 개미·벌 사진 244장으로 다시 학습합니다. 같은 사진으로 처음부터 학습한 모델과 나란히 비교합니다.

**이번 시간에 반복할 패턴:** `models.resnet18(weights=...)` · 사전학습 때와 같은 전처리 · `topk` · `ImageFolder` · 증강 변환 · `requires_grad = False` · `model.fc` 교체.

**대응 이론:** [Ch08 CNN의 정확도 높이기](https://ralbu85.github.io/lecture_deeplearning/chapters/ch08.html)

**Colab 설정:** 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**를 고른 뒤 시작합니다.

| 표시 | 뜻 |
|---|---|
| ▶ | 그대로 실행합니다 |
| ✍️ | 보여 준 코드를 손으로 타이핑해서 실행합니다. 빈칸 `___`은 채웁니다 |
| 💬 | 결과를 함께 읽습니다 |
| ✅ | 이 숫자가 나오면 맞습니다 |
| 📝 | 과제. 힌트를 보고 스스로 풉니다 |

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요.
import os, zipfile, urllib.request
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from PIL import Image, ImageOps

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
print(device)

✅ Colab에서 GPU 런타임이면 `cuda`가 나옵니다.

---

## 1. 미리 학습된 모델을 불러온다

`torchvision.models`에는 7장의 ResNet이 들어 있습니다. `weights`에 ImageNet으로 학습한 가중치를 지정하면 가중치까지 내려받습니다. `weights.meta['categories']`는 1,000개 클래스의 이름입니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
weights = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=weights).to(device)
names = weights.meta['categories']
print(len(names), names[:5])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — `1000`과 `['tench', 'goldfish', 'great white shark', 'tiger shark', 'hammerhead']`이면 맞습니다.

사전학습 모델에는 학습할 때와 같은 형태의 입력을 넣습니다. 이 모델은 224×224 사진을 ImageNet 사진의 채널별 평균과 표준편차로 표준화해 학습했습니다. `Resize(256)`으로 짧은 변을 256으로 줄이고 `CenterCrop(224)`로 가운데를 잘라 냅니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

---

## 2. 학습 없이 판별한다

실습 6과 같은 고양이 사진을 씁니다.

In [ ]:
# ▶ 그대로 실행하세요 — 예시 사진 불러오기
from skimage import data
photo = Image.fromarray(data.chelsea())
plt.imshow(photo)
plt.axis('off')
plt.show()

사진을 변환해 모델에 넣고, 1,000개 확률 가운데 큰 것 다섯 개를 봅니다. `topk(5)`는 큰 값 다섯 개와 그 위치를 돌려줍니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
model.eval()
x = tf(photo).unsqueeze(0).to(device)
with torch.no_grad():
    prob = torch.softmax(model(x), dim=1)[0]
top_p, top_i = prob.topk(5)
for p, i in zip(top_p.tolist(), top_i.tolist()):
    print(names[i], round(p, 3))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — 1~3위가 `Egyptian cat`, `tiger cat`, `tabby`이면 맞습니다. 셋 다 고양이 종류입니다.

💬 **결과 해설** — 이 모델은 우리가 한 번도 학습시키지 않았습니다. 그래도 고양이 종류 셋을 맨 위에 놓습니다. 실습 6의 모델은 같은 사진을 `cat`, `dog`, `frog` 사이에서 오갔습니다.

사진 한 장을 받아 상위 다섯 개를 돌려주는 함수로 조립합니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
def top5(photo):
    x = tf(photo).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        prob = torch.softmax(model(x), dim=1)[0]
    top_p, top_i = prob.topk(5)
    return [(names[i], round(p, 3)) for p, i in zip(top_p.tolist(), top_i.tolist())]

print(top5(photo))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — 바로 위와 같은 다섯 개면 맞습니다.

내 사진을 올립니다. 아래 셀은 **Colab에서만** 실행됩니다. 사람은 찍지 않습니다.

In [ ]:
# ▶ 그대로 실행하세요 — (Colab) 내 사진 올리기
from google.colab import files
uploaded = files.upload()

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
for name in uploaded:
    img = ImageOps.exif_transpose(Image.open(name)).convert('RGB')
    print(name, top5(img))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

💬 클래스 이름은 영어입니다. 컵, 키보드, 신발, 휴대폰처럼 1,000개 클래스에 있는 물건은 대체로 맞힙니다. 1,000개에 없는 물건은 가장 비슷한 클래스로 답합니다.

---

## 3. 왜 맞히는가 — 첫 층의 커널

ResNet-18 첫 콘볼루션 층의 커널 64개를 그림으로 그립니다. 위는 ImageNet으로 학습한 커널(pretrained), 아래는 학습 전의 무작위 커널(random)입니다.

In [ ]:
# ▶ 그대로 실행하세요 — 커널 64개 그리기
def show_kernels(w, title):
    k = w.detach().cpu()
    lo = k.amin(dim=(1, 2, 3), keepdim=True)
    hi = k.amax(dim=(1, 2, 3), keepdim=True)
    k = (k - lo) / (hi - lo)
    fig, axes = plt.subplots(4, 16, figsize=(12, 3))
    for ax, ki in zip(axes.flat, k):
        ax.imshow(ki.permute(1, 2, 0))
        ax.axis('off')
    fig.suptitle(title)
    plt.show()

show_kernels(model.conv1.weight, 'pretrained')
show_kernels(models.resnet18(weights=None).conv1.weight, 'random')

💬 **결과 해설** — 학습한 커널은 여러 방향의 줄무늬와 색 덩어리입니다. 줄무늬 커널은 같은 방향의 에지에서 큰 값을 냅니다. 무작위 커널은 잡음뿐입니다. 사진 128만 장으로 에지 같은 공통된 패턴을 이미 배워 두었기 때문에 학습 없이도 맞힙니다.

---

## 4. 내 클래스의 사진 — 개미와 벌

이번에는 1,000개 클래스 대신 내가 정한 클래스로 분류합니다. 개미와 벌 사진을 내려받습니다. 클래스마다 폴더가 하나씩 있습니다.

```
hymenoptera_data/
  train/   ants/ … bees/ …
  val/     ants/ … bees/ …
```

In [ ]:
# ▶ 그대로 실행하세요 — 개미·벌 사진 내려받기(약 45MB)
ROOT = './data/hymenoptera_data'
if not os.path.isdir(ROOT):
    os.makedirs('./data', exist_ok=True)
    urllib.request.urlretrieve('https://download.pytorch.org/tutorial/hymenoptera_data.zip',
                               './data/hymenoptera_data.zip')
    with zipfile.ZipFile('./data/hymenoptera_data.zip') as z:
        z.extractall('./data')
print(sorted(os.listdir(ROOT)))

✅ `['train', 'val']`이 나오면 됩니다.

`datasets.ImageFolder`는 폴더 이름을 클래스로 읽습니다. 변환 없이 먼저 읽어 봅니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
raw = datasets.ImageFolder(ROOT + '/train')
print(raw.classes, len(raw))
img, label = raw[0]
print(img.size, label)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — `['ants', 'bees'] 244`이면 맞습니다.

💬 **결과 해설** — 폴더 이름이 클래스 이름이 되고, 알파벳 순서로 `ants`가 0, `bees`가 1입니다. 사진은 PIL 그림 그대로이고 크기가 제각각입니다. 그래서 1절의 `Resize`와 `CenterCrop`으로 크기를 맞춥니다.

In [ ]:
# ▶ 그대로 실행하세요 — 개미 4장, 벌 4장
fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for ax, i in zip(axes, [0, 1, 2, 3, len(raw) - 4, len(raw) - 3, len(raw) - 2, len(raw) - 1]):
    img, label = raw[i]
    ax.imshow(img)
    ax.set_title(raw.classes[label])
    ax.axis('off')
plt.show()

훈련 사진에 쓸 변환에 이론 8장의 증강을 넣습니다. `RandomHorizontalFlip()`은 절반의 확률로 좌우를 뒤집고, `RandomRotation(15)`는 −15°에서 15° 사이로 돌리고, `ColorJitter`는 밝기·대비·채도를 30% 안에서 바꿉니다. 회전을 `CenterCrop` 앞에 두어 회전으로 생긴 모서리가 잘려 나가게 합니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
tf_train = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomRotation(15),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

같은 사진에 증강을 여덟 번 적용해 봅니다. `ToTensor` 앞의 다섯 단계만 써서 그림으로 봅니다.

In [ ]:
# ▶ 그대로 실행하세요 — 같은 사진, 증강 여덟 번
show_tf = transforms.Compose(tf_train.transforms[:5])
img, label = raw[0]
fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for ax in axes:
    ax.imshow(show_tf(img))
    ax.axis('off')
plt.show()

💬 **결과 해설** — 같은 사진이 부를 때마다 다르게 바뀝니다. 학습 중에는 에폭마다 이런 새 변형이 들어갑니다.

데이터셋과 로더를 만듭니다. 증강은 훈련 사진에만 쓰고, 검증 사진에는 1절의 `tf`를 씁니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
train_ds = datasets.ImageFolder(ROOT + '/train', transform=tf_train)
val_ds = datasets.ImageFolder(ROOT + '/val', transform=tf)

torch.manual_seed(42)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)
print(len(train_ds), len(val_ds))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — `244 153`이면 맞습니다.

---

## 5. 특성 학습부를 동결하고 분류부를 갈아 끼운다

### 5-1. 분류부 확인

이론 8장에서 CNN을 특성 학습부와 분류부로 나누었습니다. ResNet-18에서는 맨 끝의 `model.fc` 한 층이 분류부이고, 그 앞의 층이 모두 특성 학습부입니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
print(model.fc)
print(model.fc.in_features)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — `Linear(in_features=512, out_features=1000, bias=True)`와 `512`이면 맞습니다.

💬 **결과 해설** — 특성 학습부가 사진 한 장을 특성 512개로 바꾸고, 분류부가 그 512개로 1,000개 클래스의 점수를 냅니다. 개미와 벌을 가르려면 분류부를 출력 2개짜리로 바꿔야 합니다.

### 5-2. 특성 학습부 동결

실습 3에서 `requires_grad`가 `True`인 텐서만 `backward()` 때 기울기가 계산된다고 했습니다. 기울기가 없으면 옵티마이저도 값을 바꾸지 못합니다. 그래서 파라미터의 `requires_grad`를 `False`로 바꾸면 그 파라미터는 학습 중에도 그대로 고정됩니다. 이것이 동결입니다. 먼저 모델의 모든 파라미터를 동결합니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
for p in model.parameters():
    p.requires_grad = False

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(n_train)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — `0`이면 맞습니다.

💬 **결과 해설** — 지금은 분류부까지 모두 동결되어 학습할 파라미터가 하나도 없습니다.

### 5-3. 분류부 갈아 끼우기

`model.fc`에 새 층을 넣으면 분류부가 바뀝니다. 입력은 특성 512개 그대로이고, 출력은 개미와 벌 2개입니다. 새로 만든 층은 CPU에 있으므로 모델을 다시 `device`로 옮깁니다. 바꾼 뒤 고양이 사진을 넣어 출력 모양을 봅니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
model.fc = nn.Linear(512, 2)
model = model.to(device)
print(model.fc)

model.eval()
with torch.no_grad():
    print(model(tf(photo).unsqueeze(0).to(device)).shape)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — `Linear(in_features=512, out_features=2, bias=True)`와 `torch.Size([1, 2])`이면 맞습니다.

💬 **결과 해설** — 2절에서는 같은 사진에 점수가 1,000개 나왔는데, 이제는 2개만 나옵니다. 분류부만 바뀌었고 특성 학습부는 그대로입니다.

학습할 파라미터를 다시 셉니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in model.parameters())
print(n_train, n_all)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — `1026 11177538`이면 맞습니다.

💬 **결과 해설** — 새로 만든 층은 `requires_grad`가 기본값 `True`라서 분류부의 1,026개만 학습 대상이 됩니다. 이론 8장의 계산 512 × 2 + 2 = 1,026과 같습니다. 순서가 중요합니다. 분류부를 먼저 바꾸고 나서 동결 반복문을 돌리면 새 분류부까지 동결되어 학습할 파라미터가 0이 됩니다.

---

## 6. 학습시켜 비교한다

학습 함수와 평가 함수는 실습 6과 같습니다.

In [ ]:
# ▶ 그대로 실행하세요 — 실습 6과 같은 학습·평가 함수
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total = 0.0
    n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(yb)
        n += len(yb)
    return total / n

def evaluate(model, loader, criterion):
    model.eval()
    total = 0.0
    correct = 0
    n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            total += criterion(out, yb).item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            n += len(yb)
    return total / n, correct / n

옵티마이저에는 학습할 분류부 `model.fc`의 파라미터만 넘깁니다. 학습 전의 검증 정확도를 먼저 봅니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)
print(evaluate(model, val_loader, criterion))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — 정확도가 **0.5** 안팎이면 맞습니다. 새 분류부는 아직 무작위라 둘 중 하나를 찍는 수준입니다.

학습하기 전에 특성 학습부의 첫 층 커널과 분류부의 가중치를 복사해 둡니다. 학습이 끝난 뒤 무엇이 바뀌었는지 비교합니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
conv1_before = model.conv1.weight.clone()
fc_before = model.fc.weight.clone()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

5 에폭 학습하며 검증 정확도를 기록합니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
hist_tl = []
for epoch in range(5):
    train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    hist_tl.append(val_acc)
    print(epoch, round(val_loss, 4), round(val_acc, 4))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — 마지막 정확도가 **0.92~0.95**이면 맞습니다. 첫 에폭부터 0.7을 넘습니다.

복사해 둔 값과 비교합니다. `torch.equal`은 두 텐서의 값이 모두 같으면 `True`를 돌려줍니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
print(torch.equal(conv1_before, model.conv1.weight))
print(torch.equal(fc_before, model.fc.weight))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — `True`와 `False`이면 맞습니다.

💬 **결과 해설** — 동결한 첫 층의 커널은 5 에폭을 학습한 뒤에도 그대로이고, 분류부의 가중치만 바뀌었습니다.

같은 ResNet-18을 `weights=None`으로 만들면 가중치가 무작위인 모델이 됩니다. 같은 사진으로 처음부터 학습합니다. 이번에는 모든 파라미터를 학습하고, 에폭을 10으로 늘립니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
torch.manual_seed(42)
scratch = models.resnet18(weights=None)
scratch.fc = nn.Linear(512, 2)
scratch = scratch.to(device)
optimizer2 = torch.optim.Adam(scratch.parameters(), lr=0.001)

hist_sc = []
for epoch in range(10):
    train_one_epoch(scratch, train_loader, criterion, optimizer2)
    val_loss, val_acc = evaluate(scratch, val_loader, criterion)
    hist_sc.append(val_acc)
    print(epoch, round(val_loss, 4), round(val_acc, 4))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

💬 **결과 해설** — 검증 정확도가 0.4에서 0.8 사이를 오르내립니다. 구조도 사진도 같고, 다른 것은 가중치의 출발점뿐입니다.

In [ ]:
# ▶ 그대로 실행하세요 — 두 모델의 검증 정확도
plt.plot(range(1, 6), hist_tl, marker='o', label='pretrained + freeze')
plt.plot(range(1, 11), hist_sc, marker='o', label='from scratch')
plt.xlabel('epoch')
plt.ylabel('val accuracy')
plt.ylim(0.4, 1.0)
plt.legend()
plt.show()

검증 사진 24장에 각 모델의 판별을 적습니다. 맞으면 초록, 틀리면 빨강이고, 그림 제목에 맞힌 수가 나옵니다.

In [ ]:
# ▶ 그대로 실행하세요 — 검증 사진 24장 판별
raw_val = datasets.ImageFolder(ROOT + '/val')
view = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])
idx = list(range(0, 153, 6))[:24]
for m, title in [(model, 'pretrained + freeze'), (scratch, 'from scratch')]:
    m.eval()
    fig, axes = plt.subplots(2, 12, figsize=(20, 4))
    hit = 0
    for ax, i in zip(axes.flat, idx):
        img, label = raw_val[i]
        with torch.no_grad():
            p = m(tf(img).unsqueeze(0).to(device)).argmax(1).item()
        hit += int(p == label)
        ax.imshow(view(img))
        ax.set_title(raw_val.classes[p], color='green' if p == label else 'red')
        ax.axis('off')
    fig.suptitle(f'{title}: {hit}/24')
    plt.show()

💬 **결과 해설** — 사전학습 모델은 24장 가운데 한두 장만 틀립니다. 처음부터 학습한 모델은 그보다 몇 배 많이 틀리고, 틀리는 장수는 학습할 때마다 달라집니다. 사진 244장은 처음부터 학습하기에는 적지만, 분류부 1,026개를 학습하기에는 충분합니다.

학습한 개미·벌 모델로 사진을 판별해 봅니다. 사진 한 장을 받아 두 클래스의 확률을 돌려주는 함수를 만들고, 검증 사진 한 장으로 먼저 확인합니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
def predict(photo):
    x = tf(photo).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        prob = torch.softmax(model(x), dim=1)[0]
    return {c: round(p, 3) for c, p in zip(train_ds.classes, prob.tolist())}

img, label = raw_val[0]
print(raw_val.classes[label], predict(img))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — 실제 클래스 `ants`와 함께 `{'ants': 0.667, 'bees': 0.333}`처럼 `ants`의 확률이 더 크게 나오면 맞습니다.

이제 내 사진을 올립니다. 아래 셀은 **Colab에서만** 실행됩니다. 인터넷에서 개미나 벌 사진을 내려받아 올려 봅니다.

In [ ]:
# ▶ 그대로 실행하세요 — (Colab) 사진 올리기
from google.colab import files
uploaded = files.upload()

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
for name in uploaded:
    img = ImageOps.exif_transpose(Image.open(name)).convert('RGB')
    print(name, predict(img))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

💬 사진 이름 옆에 `ants`와 `bees`의 확률이 나옵니다. 개미나 벌이 아닌 사진을 올려도 둘 중 하나의 확률이 높게 나옵니다. 분류부의 출력이 2개뿐이라 둘 중에서만 고르기 때문입니다.

---

## 7. 실습 과제 (8분)

### 📝 과제 1 · 사진을 더 줄이면

훈련 사진을 클래스마다 20장, 모두 40장만 써서 동결 학습을 다시 합니다. 새 모델을 만들어 5절과 같이 동결하고, 10 에폭 학습한 뒤 검증 정확도를 적습니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: train_ds.targets에서 라벨이 0인 앞 20개와 1인 앞 20개의 번호를 골라 Subset(train_ds, idx)로 데이터셋을 만들고, DataLoader(batch_size=8, shuffle=True)로 감쌉니다.
# 힌트: 모델은 models.resnet18(weights=weights)로 새로 만들고, 5절의 동결·교체 코드와 6절의 학습 반복문을 그대로 씁니다.

✏️ **클래스당 20장일 때 검증 정확도:**


### 📝 과제 2 · 모든 층을 학습하면

동결하지 않고 모든 파라미터를 학습합니다. 사전학습 가중치에서 출발하므로 학습률을 0.0001로 작게 둡니다. 5 에폭 학습한 뒤 검증 정확도를 동결 학습과 비교합니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: models.resnet18(weights=weights)로 새로 만들고 fc만 nn.Linear(512, 2)로 바꿉니다. 동결 반복문은 쓰지 않습니다.
# 힌트: 옵티마이저에 모델 전체의 파라미터를 넘기고 lr=0.0001로 둡니다.

✏️ **모든 층을 학습했을 때의 검증 정확도와 동결 학습과의 차이:**


---

## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 사전학습 모델 불러오기 | `models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)`. 클래스 이름은 `weights.meta['categories']` |
| 같은 형태의 입력 | `Resize(256)` → `CenterCrop(224)` → `ToTensor` → ImageNet 평균·표준편차로 `Normalize` |
| 학습 없이 판별 | `softmax` 뒤 `topk(5)`로 상위 다섯 클래스 |
| 첫 층의 커널 | 학습한 커널은 에지와 색, 무작위 커널은 잡음 |
| 내 클래스의 데이터 | `datasets.ImageFolder`. 폴더 이름이 클래스, 알파벳 순서로 번호 |
| 증강 | `RandomHorizontalFlip`, `RandomRotation`, `ColorJitter`. 훈련 사진에만 |
| 동결 | `requires_grad = False`면 기울기가 없어 학습 중에도 값이 그대로. 학습 뒤에도 첫 층 커널은 `torch.equal`로 `True` |
| 분류부 교체 | 동결한 뒤 `model.fc = nn.Linear(512, 2)`. 출력이 1,000개에서 2개로, 학습 파라미터는 1,026개 |
| 결과 | 사진 244장으로 동결 학습 약 0.93, 처음부터 학습 0.7 안팎 |
| 사진 판별 | `predict` 함수로 두 클래스의 확률. Colab에서 `files.upload()`로 올린 사진에도 그대로 |

**다음:** 객체탐지에서는 사진 속 물체가 무엇인지에 더해 어디에 있는지까지 찾습니다.